# Binary Classification on Tabular Data - Predicting Heart Disease (PyTorch)

This is the PyTorch version of the Keras [course notebook](course_notebook.ipynb). The data loading and preprocessing are identical — only the model building, training, and evaluation sections differ.

## Introduction

This notebook shows how to set up and train a Neural Network model for *binary classification*, when the dataset is *tabular* (rather than unstructured data like images or text) and has a mix of numeric and categorical features. Since tabular datasets are often made available in CSV files, the notebook demonstrates the full CSV-to-trained-model workflow.




### The dataset

The dataset ([more background on the data](https://archive.ics.uci.edu/ml/datasets/heart+Disease)) has information on 303 patients, one in each row. Each column (i.e., feature) contains information on a particular attribute of the patient. The column named "Target" indicates if the patient has been diagnosed with heart disease or not and is the label (i.e., the dependent variable) that we want to predict using the other columns.

Feature description (copied from [here](https://keras.io/examples/structured_data/structured_data_classification_from_scratch/)):

Column| Description| Feature Type
------------|--------------------|----------------------
Age | Age in years | Numerical
Sex | (1 = male; 0 = female) | Categorical
CP | Chest pain type (0, 1, 2, 3, 4) | Categorical
Trestbpd | Resting blood pressure (in mm Hg on admission) | Numerical
Chol | Serum cholesterol in mg/dl | Numerical
FBS | fasting blood sugar in 120 mg/dl (1 = true; 0 = false) | Categorical
RestECG | Resting electrocardiogram results (0, 1, 2) | Categorical
Thalach | Maximum heart rate achieved | Numerical
Exang | Exercise induced angina (1 = yes; 0 = no) | Categorical
Oldpeak | ST depression induced by exercise relative to rest | Numerical
Slope | Slope of the peak exercise ST segment | Numerical
CA | Number of major vessels (0-3) colored by fluoroscopy | Both numerical & categorical
Thal | 3 = normal; 6 = fixed defect; 7 = reversible defect | Categorical
Target | Diagnosis of heart disease (1 = true; 0 = false) | Target

## Technical preliminaries

Throughout the course, we will load the following packages as the first step.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

When we train Deep Learning models, randomness enters the process in a few different places.
*   Starting values for the weights (the optimizer will try to improve these weights)
*   The order in which we process the minibatches when we do SGD
*   When we split the data into Train, Validation, Test etc
*   Dropout (if we use regularization)

We next set the seed for the different random number generators so that the
results will be the same every time the notebook is run. 🤞


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## Read in the data

Conveniently, the dataset in CSV form has been made available online (by [Francois Chollet](https://twitter.com/fchollet)) and we can load it into a Pandas dataframe with the very useful `pd.read_csv` command.

In [ ]:
df = pd.read_csv("http://storage.googleapis.com/download.tensorflow.org/data/heart.csv")

In [ ]:
df.shape

The dataset has 303 rows and 14 columns (13 independent variables + 1 dependent variable):

Let's take a look at the first few rows:

In [ ]:
df.head()

Let's take a quick look to see if the 1s and 0s are balanced.

In [ ]:
df.target.value_counts(normalize=True, dropna=False)

It is a bit imbalanced.

What's a 'naive' **baseline model** for this problem?

<br> <br> <br> <br>

A baseline model would be to just predict a probability of 0.0 for every patient. That will result in 72.6% accuracy.

Any fancy model we build needs to do better than this.

We will come back to this later.

## Preprocessing

This dataset has both categorical variables and numeric variables.

It will be convenient (for later processing) to collect these groups of variables into two lists.

In [ ]:
categorical_variables = ['sex', 'cp', 'fbs', 'restecg','exang', 'ca', 'thal']
numerics = ['age', 'trestbps','chol', 'thalach', 'oldpeak', 'slope']

NNs require all their inputs to be numeric so we will first preprocess this raw data as follows:
- *one-hot encode* the categorical variables
- *normalize* the numeric variables


With the pandas `get_dummies` function, you can one-hot-encode in one line.


In [ ]:
df = pd.get_dummies(df, columns = categorical_variables)

In [ ]:
df.head()


NNs work best when the inputs are all roughly in the same range. So standard practice is to **standardize** the numeric variables.

Before we do so, let's split the data into an 80% training set and 20% test set (*why should we split **before** normalization?*).

In [ ]:
test_df = df.sample(frac=0.2, random_state=42)
train_df = df.drop(test_df.index)

In [ ]:
train_df.shape

In [ ]:
test_df.shape

OK, let's calculate the mean and standard deviation of every numeric variable in the training set.

In [ ]:
means = train_df[numerics].mean()
sd = train_df[numerics].std()

In [ ]:
means

Let's standardize the train and test dataframes with these means and standard deviations.

In [ ]:
train_df[numerics]= (train_df[numerics] - means)/sd

In [ ]:
test_df[numerics]= (test_df[numerics] - means)/sd

In [ ]:
train_df.head()

At this point, the data is entirely numeric.

The easiest way to feed data to PyTorch is as tensors, so we first convert our dataframes to NumPy arrays and then to PyTorch tensors. We use `.astype(float)` because `get_dummies` can leave some columns as `object` dtype (from the original string values in `thal`), and PyTorch requires numeric arrays.

In [ ]:
train = train_df.to_numpy().astype(float)
test = test_df.to_numpy().astype(float)

Final step: Our features $X$ and dependent variable $y$ are both inside the `train` and `test` arrays so let's separate them out.

Note that the `target` column is our $y$ variable and it is column #6 from the left (counting from 0).

The `np.delete` function is perfect for selecting all columns except one.

In [ ]:
train_X = np.delete(train, 6, axis=1)
test_X = np.delete(test, 6, axis=1)

Check that it worked.

In [ ]:
train_X.shape, test_X.shape

Next, select just the 6th column and define the train and test $y$ variables.

In [ ]:
train_y = train[:, 6]
test_y = test[:, 6]

In [ ]:
train_y.shape, test_y.shape

### Convert to PyTorch tensors

PyTorch works with its own `Tensor` type rather than NumPy arrays. We convert now and also create a `DataLoader` that will handle batching and shuffling during training.

In [ ]:
# Convert to float32 tensors
train_X_t = torch.tensor(train_X, dtype=torch.float32)
train_y_t = torch.tensor(train_y, dtype=torch.float32)
test_X_t = torch.tensor(test_X, dtype=torch.float32)
test_y_t = torch.tensor(test_y, dtype=torch.float32)

In [ ]:
# Split training data into train and validation (80/20)
n = len(train_X_t)
n_val = int(n * 0.2)
# Use the last 20% as validation (matching Keras' validation_split behavior)
val_X_t = train_X_t[-n_val:]
val_y_t = train_y_t[-n_val:]
trn_X_t = train_X_t[:-n_val]
trn_y_t = train_y_t[:-n_val]

In [ ]:
# Create a DataLoader for batching and shuffling
train_dataset = TensorDataset(trn_X_t, trn_y_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## Build a model

### Define model in PyTorch

In PyTorch, we define a model by subclassing `nn.Module`. This is equivalent to the Keras functional API.

* We will start with a single hidden layer.
* Since this is a *binary classification problem*, we will use a sigmoid activation in the output layer.

In PyTorch, we define the layers in `__init__` and specify how data flows through them in `forward`.

In [ ]:
num_columns = train_X.shape[1]

class HeartDiseaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),  # Hidden layer: input_dim -> 16
            nn.ReLU(),
            nn.Linear(16, 1),          # Output layer: 16 -> 1
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = HeartDiseaseNet(num_columns)

We can print the model to see its structure (similar to `model.summary()` in Keras).

In [ ]:
print(model)

To count parameters in PyTorch, we iterate over `model.parameters()`.

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

Let's hand-calculate the number of parameters to verify.

In [ ]:
(29 + 1) * 16 + (16 + 1) * 1

We can also inspect each layer's weight shapes directly.

In [ ]:
for name, param in model.named_parameters():
    print(f"{name:20s} {str(list(param.shape)):15s} ({param.numel()} params)")

### Set optimization parameters

In PyTorch, we set up the loss function and optimizer as separate objects (rather than calling `model.compile` as in Keras).

*   **Loss function**: `nn.BCELoss()` — Binary Cross-Entropy, equivalent to Keras' `binary_crossentropy`
*   **Optimizer**: `torch.optim.Adam` — the same Adam optimizer we used in Keras

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Train the model

Unlike Keras where `model.fit()` handles the entire training loop, in PyTorch we write the training loop explicitly. This gives us full control over what happens at each step:

1. **Forward pass**: compute predictions
2. **Compute loss**: compare predictions to targets
3. **Backward pass**: compute gradients
4. **Update weights**: optimizer takes a step

We'll train for 300 epochs with a batch size of 32, matching the Keras notebook.

In [ ]:
num_epochs = 300

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(num_epochs):
    # --- Training phase ---
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0

    for xb, yb in train_loader:
        optimizer.zero_grad()           # reset gradients
        preds = model(xb)               # forward pass
        loss = criterion(preds, yb)     # compute loss
        loss.backward()                 # backward pass (compute gradients)
        optimizer.step()                # update weights

        epoch_loss += loss.item() * len(xb)
        epoch_correct += ((preds > 0.5).float() == yb).sum().item()
        epoch_total += len(xb)

    train_loss = epoch_loss / epoch_total
    train_acc = epoch_correct / epoch_total

    # --- Validation phase ---
    model.eval()
    with torch.no_grad():
        val_preds = model(val_X_t)
        val_loss = criterion(val_preds, val_y_t).item()
        val_acc = ((val_preds > 0.5).float() == val_y_t).float().mean().item()

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

Let's take a moment to understand the numbers being reported.


---


Plotting metrics like loss and accuracy as a function of the # of epochs is a good way to understand how training has progressed.

In [ ]:
loss_values = history["loss"]
val_loss_values = history["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "bo", label="Training loss", markersize=2)
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

Do you think there's overfitting?

<br> <br> <br> <br> <br> <br> <br> <br>



There does seem to be overfitting.

If there's overfitting at epoch N, we could go back, re-initialize the model and just run it for  N epochs - that would an example of **early stopping**.


Let's look at the accuracy curves as well.


In [ ]:
plt.clf()
acc = history["accuracy"]
val_acc = history["val_accuracy"]
plt.plot(epochs, acc, "bo", label="Training acc", markersize=2)
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Evaluate the model

Let's see **how well the model does on the test set**.

In PyTorch, we compute predictions manually and calculate loss and accuracy ourselves (there's no built-in `model.evaluate` like in Keras).

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(test_X_t)
    test_loss = criterion(test_preds, test_y_t).item()
    test_acc = ((test_preds > 0.5).float() == test_y_t).float().mean().item()

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

How does the accuracy of this "neural model" compare to the accuracy of our baseline model?

The baseline model had an accuracy of 72.6% so our first NN model is certainly beating it.

## PyTorch vs Keras: key differences

| | Keras | PyTorch |
|---|---|---|
| **Model definition** | Functional API or Sequential | Subclass `nn.Module` |
| **Training** | `model.fit()` (one line) | Explicit loop (forward, loss, backward, step) |
| **Evaluation** | `model.evaluate()` | Manual with `torch.no_grad()` |
| **Optimizer setup** | `model.compile()` | Separate `torch.optim` object |
| **Train/eval mode** | Automatic | Must call `model.train()` / `model.eval()` |
| **Gradient computation** | Automatic | Must call `loss.backward()` and `optimizer.zero_grad()` |

PyTorch requires more code but gives you full visibility into the training process. This is especially valuable when debugging or implementing custom training logic.

## Saving and loading the model

In PyTorch, the standard approach is to save just the model's learned weights (called `state_dict`) rather than the entire model object. To use the model later, you reconstruct the architecture and load the saved weights.

```python
# Save
torch.save(model.state_dict(), "heart_model.pth")

# Load
model = HeartDiseaseNet(num_columns)
model.load_state_dict(torch.load("heart_model.pth"))
model.eval()
```

As with the Keras notebook, we did pre-processing (one-hot encoding and normalization) *outside* the model, so you'd need to carry that information along with the saved weights for inference on new data.